# Phase 2 — Synthetic Data + Pretraining (Colab)
**DocLayout-YOLO-Indic**

Runs the Phase 2 data engine and pretraining on Colab. Key design choices for Colab:
- **Code** lives in Drive (`MyDrive/doclayout-yolo-indic`), same as Phase 1.
- **Generation writes to LOCAL `/content`** (fast), then we tar the ~7 GB corpus and copy *one* file to Drive. (Never write 150K small files straight to Drive — it is slow and hits file-count limits.)
- **Data generation** = T4 or any runtime (CPU-bound, parallelised). **Pretraining** = A100.

Runtime: `Runtime → Change runtime type` → **T4** for Cells 0–5, switch to **A100** for Cell 6.

## Cell 0 — Bootstrap (mount Drive, install data-engine deps)

In [ ]:
import os, sys, subprocess, json
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=True)
PROJECT_ROOT = Path('/content/drive/MyDrive/doclayout-yolo-indic')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT))

# Data-engine deps only (CPU). Training deps installed later in Cell 6.
subprocess.run(['pip','install','-q','uharfbuzz','fonttools','pillow','numpy','tqdm'], check=True)

import torch
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
print('vCPUs   :', os.cpu_count())
print('PROJECT_ROOT:', PROJECT_ROOT)

## Cell 1 — Get the Phase 2 code from GitHub
Clones your repo, unzips the code bundle, and syncs it into `PROJECT_ROOT` so the rest of the notebook works unchanged. Re-run anytime to pull the latest.

In [ ]:
import subprocess, shutil, zipfile
from pathlib import Path

GITHUB_URL    = 'https://github.com/<your-username>/mtech-project-aiml.git'  # set this
GITHUB_BRANCH = 'main'
REPO_LOCAL    = Path('/content/_repo')

# clone or pull
if (REPO_LOCAL / '.git').exists():
    subprocess.run(['git','-C',str(REPO_LOCAL),'pull','--ff-only'], check=True)
else:
    shutil.rmtree(REPO_LOCAL, ignore_errors=True)
    subprocess.run(['git','clone','--depth','1','-b',GITHUB_BRANCH,
                    GITHUB_URL, str(REPO_LOCAL)], check=True)

# unzip the code bundle committed under files/
zip_path = REPO_LOCAL / 'files' / 'doclayout-yolo-indic.zip'
assert zip_path.exists(), f'Zip not found at {zip_path}'
unzip_dir = Path('/content/_unzipped')
shutil.rmtree(unzip_dir, ignore_errors=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(unzip_dir)

# folder that CONTAINS src/ (parent.parent of src/config.py)
src_root = next(p.parent.parent for p in unzip_dir.rglob('src/config.py'))
print('src_root:', src_root)

# sync into PROJECT_ROOT (Drive) so all other cells work unchanged
for item in ['src','tests','requirements.txt','README.md']:
    s = src_root / item; d = PROJECT_ROOT / item
    if s.is_dir():
        shutil.rmtree(d, ignore_errors=True); shutil.copytree(s, d)
    elif s.exists():
        shutil.copy(s, d)

assert (PROJECT_ROOT / 'src' / 'synthetic_data' / 'generator.py').exists()
print('Code synced to', PROJECT_ROOT, '✓')

## Cell 2 — Fonts (downloads ~5 MB of Noto fonts into Drive, cached)

In [ ]:
%cd {PROJECT_ROOT}
!python -m src.synthetic_data.font_setup

## Cell 3 — Pilot: 200 pages + visual QC (the Week-3 quality gate)
Generates a small batch to **local disk** and shows a contact sheet inline. Eyeball it (ideally with a native reader): shirorekha present, conjuncts/matras correct, Urdu reads right-to-left, boxes tight. Do not proceed to full generation until this looks right.

In [ ]:
%cd {PROJECT_ROOT}
!python -m src.synthetic_data.parallel --num 200 --tag pilot --out /content/pilot --workers $(nproc)

# inline contact sheet from the LOCAL pilot dir
import json
from pathlib import Path
from PIL import Image
from src.synthetic_data.quality_inspector import overlay
root = Path('/content/pilot')
thumbs=[]
for i in range(1, 13):
    rec = json.loads((root/'annotations'/f'pilot_{i:06d}.json').read_text())
    img = overlay(root/'images'/rec['image_file'], rec['annotations'])
    thumbs.append(img.resize((300,300)))
sheet = Image.new('RGB',(4*300,3*300),'white')
for k,t in enumerate(thumbs):
    r,c=divmod(k,4); sheet.paste(t,(c*300,r*300))
sheet.save('/content/pilot_contact_sheet.png')
from IPython.display import Image as IPImage, display
display(IPImage('/content/pilot_contact_sheet.png'))

## Cell 4 — Generate the corpus (RESUMABLE; shards saved to Drive)
CPU-bound — the **GPU is unused**, so run this on a cheap high-RAM/T4 runtime, not the A100. ~1.1 s/page single-core → roughly **6 h for 150K on 8 vCPUs**.

Generation happens in shards; each finished shard is saved to `shards/` on Drive, so a disconnect never loses completed work. **If it disconnects, just re-run this same cell** — it skips finished shards and continues.

Start with `N = 30000` for the pilot, then set `150000`. Save the A100 for Cell 6.

In [ ]:
%cd {PROJECT_ROOT}
from pathlib import Path
from src.synthetic_data.resumable import generate_resumable

N = 150000                       # set 30000 for the pilot first
SHARD_DIR = PROJECT_ROOT / 'shards'        # on Drive — survives disconnects

generate_resumable(
    num_docs=N, tag='indicsynth',
    work_dir=Path('/content/work'),         # local scratch (ephemeral, fine)
    drive_dir=SHARD_DIR,                     # finished shards land here
    shard_size=5000,                         # risk window if it dies mid-shard
    workers=0,                               # 0 = use all vCPUs
)
print('Done. Shards on Drive:', len(list(SHARD_DIR.glob('indicsynth_shard_*.tar'))))

## Cell 4b — Upload corpus to HuggingFace, then free up storage
Uploads the `shards/` corpus to a HF dataset (your durable store), **verifies** every shard arrived, then deletes the local scratch and — only after a verified upload — the Drive copy, reclaiming Drive space. Run this *after* Cell 4 has produced all shards. Needs a HF **write** token.

In [ ]:
!pip install -q huggingface_hub
import os, shutil
from pathlib import Path
from huggingface_hub import HfApi, login

HF_TOKEN = ''                                  # paste a write token
HF_REPO  = 'your-username/indicsynth-150k'      # change to your namespace
FREE_DRIVE_AFTER_UPLOAD = True                  # delete Drive shards once verified
HF_TOKEN = HF_TOKEN or os.environ.get('HF_TOKEN','')
assert HF_TOKEN, 'Set a HuggingFace write token first'

SHARD_DIR = PROJECT_ROOT / 'shards'
local_tars = sorted(SHARD_DIR.glob('indicsynth_shard_*.tar'))
assert local_tars, f'No shards in {SHARD_DIR} — run Cell 4 first'
print(f'{len(local_tars)} shards to upload')

# 1) upload
login(token=HF_TOKEN)
api = HfApi()
api.create_repo(HF_REPO, repo_type='dataset', private=True, exist_ok=True)
api.upload_folder(folder_path=str(SHARD_DIR), path_in_repo='shards',
                  repo_id=HF_REPO, repo_type='dataset')

# 2) verify every shard is present on HF before deleting anything
remote = {Path(f).name for f in api.list_repo_files(HF_REPO, repo_type='dataset')
          if f.endswith('.tar')}
missing = [t.name for t in local_tars if t.name not in remote]
assert not missing, f'Upload incomplete, NOT cleaning up. Missing: {missing}'
print(f'Verified all {len(local_tars)} shards on HF -> '
      f'https://huggingface.co/datasets/{HF_REPO}')

# 3) free storage (only runs after verification passed)
shutil.rmtree('/content/work', ignore_errors=True)        # local scratch
shutil.rmtree('/content/_unzipped', ignore_errors=True)
if FREE_DRIVE_AFTER_UPLOAD:
    shutil.rmtree(SHARD_DIR, ignore_errors=True)           # Drive copy
    print('Freed Drive shards. Corpus now lives on HF; Cell 5 will pull it.')
else:
    print('Kept Drive shards (set FREE_DRIVE_AFTER_UPLOAD=True to reclaim space).')

## Cell 5 — Assemble shards + convert to YOLO (CPU)
Uses the Drive `shards/` if present; otherwise downloads them from your HF dataset. Then merges into one COCO and builds YOLO labels + `data.yaml`. Idempotent — safe to re-run.

In [ ]:
%cd {PROJECT_ROOT}
from pathlib import Path
from src.synthetic_data.resumable import assemble_from_shards
from src.pretraining.train_synthetic import coco_to_yolo

SHARD_DIR = PROJECT_ROOT / 'shards'
if SHARD_DIR.exists() and list(SHARD_DIR.glob('indicsynth_shard_*.tar')):
    shards_src = SHARD_DIR
else:
    # pull from HuggingFace (set HF_REPO; add token=... if the repo is private)
    from huggingface_hub import snapshot_download
    HF_REPO = 'your-username/indicsynth-150k'
    dl = snapshot_download(repo_id=HF_REPO, repo_type='dataset',
                           allow_patterns='shards/*.tar',
                           local_dir='/content/hf_corpus')
    shards_src = Path(dl) / 'shards'
print('Using shards from:', shards_src)

coco, images = assemble_from_shards('indicsynth', shards_src,
                                    Path('/content/IndicSynth'))
yaml_path = coco_to_yolo(coco, images, Path('/content/yolo_dataset'))
print(yaml_path.read_text())

## Cell 6 — Pretrain on A100  ⚠️ switch runtime to A100 first
Installs training deps + the DocLayout-YOLO package, downloads the **DocSynth300K-pretrained** checkpoint (the correct initializer — see notes), and trains for 30 epochs.

In [ ]:
%cd {PROJECT_ROOT}
!pip install -q torch torchvision ultralytics pycocotools
!pip install -q git+https://github.com/opendatalab/DocLayout-YOLO.git

from src.pretraining.train_synthetic import download_base_checkpoint, train
from pathlib import Path
ckpt = download_base_checkpoint('/content')   # juliozhao/DocLayout-YOLO-DocSynth300K-pretrain
train(Path('/content/yolo_dataset/data.yaml'), ckpt)

## Cell 7 — Back up the pretrained checkpoint to Drive

In [ ]:
import shutil
from pathlib import Path
src = Path('output/checkpoints/doclayout_yolo_indic_pretrained.pt')
if src.exists():
    dst = PROJECT_ROOT/'output'/'checkpoints'/src.name
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(src, dst)
    print('Backed up ->', dst)
else:
    print('No checkpoint found yet — run Cell 6 on an A100 runtime.')

## Datasets for later phases (read now, act when you reach them)

**IndicDLP (Phase 3/4 — fine-tune + eval, *not* needed for Phase 2).**
Do **not** upload your 80 GB `indicdlp.tar` to Drive. It is already at a public URL, so pull it straight into Colab local disk when Phase 3 starts:
```
!wget -q https://objectstore.e2enetworks.net/indic-dlp/indicdlp.tar -O /content/indicdlp.tar
!tar -xf /content/indicdlp.tar -C /content/indicdlp
```
Re-download per session (free, ~10–20 min) rather than parking 80 GB in Drive.

**BaDLAD (Phase 3 — self-training + test).**
Set up Kaggle API access now, grab only the **labeled** split to confirm access; defer the ~4M unlabeled images until Phase 3 and use only a ~200K subset.
```
from google.colab import files; files.upload()   # upload kaggle.json
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle
# then download the BaDLAD dataset/competition data into /content
```